[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_3_5/07_tag_3_5_batchnorm_dropout.ipynb)

# Tag 3-5 - 07 Batch Normalization und Dropout

Dropout und Batch Normalization lösen unterschiedliche Probleme:

- **Dropout** wirkt als Regularisierung. Das sieht man am besten, wenn ein großes Modell mit wenig Daten overfittet.
- **Batch Normalization** stabilisiert Aktivierungen. Das sieht man am besten, wenn die Lernrate relativ aggressiv ist.

Deshalb enthält dieses Notebook zwei getrennte Demos. Das ist didaktisch klarer als ein einziger Datensatz, auf dem beide Effekte zufällig gleichzeitig sichtbar sein müssen.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


## Datensatz: Digits

Der Digits-Datensatz besteht aus 8x8-Pixelbildern handgeschriebener Ziffern. Für die Dropout-Demo verwenden wir absichtlich nur sehr wenige Trainingsbeispiele.


In [ ]:
digits = load_digits()
X = (digits.data / 16.0).astype("float32")
y = digits.target.astype("int64")

X_train_small, X_rest, y_train_small, y_rest = train_test_split(
    X, y, train_size=80, random_state=RANDOM_STATE, stratify=y
)
X_val_small, X_test_small, y_val_small, y_test_small = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest
)

fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for ax, image, label in zip(axes.ravel(), X_train_small[:12], y_train_small[:12]):
    ax.imshow(image.reshape(8, 8), cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Beispiele aus dem Digits-Datensatz")
plt.tight_layout()
plt.show()

print("Kleines Train/Val/Test:", X_train_small.shape, X_val_small.shape, X_test_small.shape)


In [ ]:
def draw_blocks(labels, title):
    fig, ax = plt.subplots(figsize=(11, 2.2))
    ax.axis("off")
    width = 0.12
    xs = np.linspace(0.02, 0.86, len(labels))
    for x, label in zip(xs, labels):
        ax.add_patch(plt.Rectangle((x, 0.35), width, 0.34, fill=False, linewidth=2))
        ax.text(x + width / 2, 0.52, label, ha="center", va="center", fontsize=8)
    for x1, x2 in zip(xs[:-1], xs[1:]):
        ax.annotate("", xy=(x2, 0.52), xytext=(x1 + width, 0.52), arrowprops=dict(arrowstyle="->", linewidth=1.6))
    ax.set_title(title)
    plt.show()


draw_blocks(["Input 64", "Dense 768", "Dense 768", "Dense 384", "output 10"], "Baseline: großes MLP")
draw_blocks(["Input 64", "Dense", "Dropout", "Dense", "Dropout", "Dense", "output"], "Dropout: zufällig deaktivierte Aktivierungen")
draw_blocks(["Input 64", "Dense", "BatchNorm", "ReLU", "Dense", "BatchNorm", "ReLU", "output"], "BatchNorm: Normalisierung vor ReLU")


## Demo 1: Dropout gegen Overfitting

Das Modell ist absichtlich viel zu groß für 80 Trainingsbilder. Ohne Dropout kann es die Trainingsdaten fast auswendig lernen. Dropout sollte die Lücke zwischen Training und Validierung verkleinern und den besten Validation Loss verbessern.


In [ ]:
class LrHistory(callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.values = []

    def on_epoch_end(self, epoch, logs=None):
        self.values.append(float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)))


def compile_digits_model(model, learning_rate=0.003):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_plain_big_model():
    return compile_digits_model(
        keras.Sequential(
            [
                keras.Input(shape=(64,), name="pixels"),
                layers.Dense(768, activation="relu", name="hidden_1"),
                layers.Dense(768, activation="relu", name="hidden_2"),
                layers.Dense(384, activation="relu", name="hidden_3"),
                layers.Dense(10, activation="softmax", name="output"),
            ],
            name="plain_big_digits_mlp",
        )
    )


def build_dropout_big_model():
    return compile_digits_model(
        keras.Sequential(
            [
                keras.Input(shape=(64,), name="pixels"),
                layers.Dense(768, activation="relu", name="hidden_1"),
                # Parameter im Fokus: 60 Prozent der Aktivierungen werden im Training deaktiviert.
                layers.Dropout(0.60, name="dropout_1"),
                layers.Dense(768, activation="relu", name="hidden_2"),
                layers.Dropout(0.60, name="dropout_2"),
                layers.Dense(384, activation="relu", name="hidden_3"),
                layers.Dense(10, activation="softmax", name="output"),
            ],
            name="dropout_big_digits_mlp",
        )
    )


dropout_histories = {}
dropout_lr_histories = {}
dropout_models = {}
for name, builder in [("plain", build_plain_big_model), ("dropout", build_dropout_big_model)]:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = builder()
    lr_history = LrHistory()
    history = model.fit(
        X_train_small,
        y_train_small,
        validation_data=(X_val_small, y_val_small),
        epochs=120,
        batch_size=16,
        verbose=0,
        callbacks=[
            callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5),
            lr_history,
        ],
    )
    test_loss, test_accuracy = model.evaluate(X_test_small, y_test_small, verbose=0)
    dropout_histories[name] = history
    dropout_lr_histories[name] = lr_history.values
    dropout_models[name] = model
    print(f"{name:>7}: Test Accuracy={test_accuracy:.1%}, bester Val Loss={min(history.history['val_loss']):.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
summary_rows = []
for name, history in dropout_histories.items():
    train_final = history.history["accuracy"][-1]
    val_final = history.history["val_accuracy"][-1]
    gap = train_final - val_final
    summary_rows.append(
        {
            "modell": name,
            "final_train_acc": train_final,
            "final_val_acc": val_final,
            "overfitting_gap": gap,
            "best_val_loss": min(history.history["val_loss"]),
        }
    )
    axes[0].plot(history.history["loss"], alpha=0.4, label=f"{name} train")
    axes[0].plot(history.history["val_loss"], linewidth=2, label=f"{name} val")
    axes[1].plot(history.history["val_accuracy"], label=name)
    axes[2].plot(dropout_lr_histories[name], label=name)
axes[0].set_title("Dropout-Demo: Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend(fontsize=8)
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
axes[2].set_title("ReduceLROnPlateau: Learning Rate")
axes[2].set_xlabel("Epoche")
axes[2].set_ylabel("Learning Rate")
axes[2].legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame(summary_rows).round(4))


## Demo 2: Batch Normalization bei aggressiver Lernrate

Für BatchNorm verwenden wir mehr Trainingsdaten und SGD mit hoher Lernrate. Der Effekt ist hier nicht "weniger Overfitting", sondern stabilere und schnellere Optimierung.


In [ ]:
X_train_bn, X_rest_bn, y_train_bn, y_rest_bn = train_test_split(
    X, y, train_size=500, random_state=RANDOM_STATE, stratify=y
)
X_val_bn, X_test_bn, y_val_bn, y_test_bn = train_test_split(
    X_rest_bn, y_rest_bn, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest_bn
)


def compile_sgd_model(model):
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.15),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_sgd_plain_model():
    return compile_sgd_model(
        keras.Sequential(
            [
                keras.Input(shape=(64,), name="pixels"),
                layers.Dense(256, activation="relu", name="hidden_1"),
                layers.Dense(256, activation="relu", name="hidden_2"),
                layers.Dense(128, activation="relu", name="hidden_3"),
                layers.Dense(10, activation="softmax", name="output"),
            ],
            name="sgd_plain_digits_mlp",
        )
    )


def build_sgd_batchnorm_model():
    return compile_sgd_model(
        keras.Sequential(
            [
                keras.Input(shape=(64,), name="pixels"),
                layers.Dense(256, use_bias=False, name="hidden_1_linear"),
                # Parameter im Fokus: BatchNorm stabilisiert die Verteilung vor ReLU.
                layers.BatchNormalization(name="batchnorm_1"),
                layers.Activation("relu", name="relu_1"),
                layers.Dense(256, use_bias=False, name="hidden_2_linear"),
                layers.BatchNormalization(name="batchnorm_2"),
                layers.Activation("relu", name="relu_2"),
                layers.Dense(128, activation="relu", name="hidden_3"),
                layers.Dense(10, activation="softmax", name="output"),
            ],
            name="sgd_batchnorm_digits_mlp",
        )
    )


batchnorm_histories = {}
for name, builder in [("plain_sgd", build_sgd_plain_model), ("batchnorm_sgd", build_sgd_batchnorm_model)]:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = builder()
    history = model.fit(
        X_train_bn,
        y_train_bn,
        validation_data=(X_val_bn, y_val_bn),
        epochs=35,
        batch_size=32,
        verbose=0,
    )
    test_loss, test_accuracy = model.evaluate(X_test_bn, y_test_bn, verbose=0)
    batchnorm_histories[name] = history
    print(f"{name:>13}: Test Accuracy={test_accuracy:.1%}, finaler Val Loss={history.history['val_loss'][-1]:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in batchnorm_histories.items():
    axes[0].plot(history.history["val_loss"], label=name)
    axes[1].plot(history.history["val_accuracy"], label=name)
axes[0].set_title("BatchNorm-Demo: Validation Loss")
axes[0].set_xlabel("Epoche")
axes[0].legend()
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoche")
axes[1].legend()
plt.tight_layout()
plt.show()
